# Phase 4 — Training and Testing an Ensemble Method: Random Forest
## Dataset: GSE6575 — Gene Expression in Blood of Children with ASD
**Dr. Nourhène Ben Rabah — Introduction to Machine Learning**

This phase introduces **Random Forest** as a second ensemble method and compares it
with the **Extra Trees** model trained in Phase 3.

- **Part 1** — Principle of Random Forest
- **Part 2** — Training and evaluation on test data
- **Part 3** — Comparison with Extra Trees (Phase 3)
- **Part 4** — Advantages and limitations analysis


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All libraries loaded successfully.")

---
## Load Data and Reproduce Phase 3 Setup

We reload the same prepared data and reproduce the **exact same** train/test split
and PCA used in Phase 3, so that the comparison is fair.


In [ ]:
# ── Load prepared data (same files as Phase 3) ───────────────────────────────
X = pd.read_csv('X_prepared.csv', index_col=0)
y = pd.read_csv('y_labels.csv', index_col=0).squeeze()

print(f"Feature matrix X : {X.shape}  (samples x probes)")
print(f"Label vector y   : {y.shape}")
print()
print("Class distribution:")
for cls, n in y.value_counts().items():
    print(f"  {cls:<35} {n} samples")

In [ ]:
# ── Reproduce Phase 3 train/test split and PCA exactly ───────────────────────
# Same random_state=42, test_size=0.20, stratify=y as Phase 3.
# This ensures we compare both models on the IDENTICAL test samples.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Train / Test split  : {X_train.shape[0]} / {X_test.shape[0]} samples")
print(f"PCA components kept : {pca.n_components_}  (explaining 95% of variance)")
print(f"X_train after PCA   : {X_train_pca.shape}")
print(f"X_test  after PCA   : {X_test_pca.shape}")

---
## Part 1 — Principle of Random Forest

### 1.1 How Random Forest Works

Random Forest is an ensemble method that builds a collection of decision trees
and combines their predictions by **majority vote**.

Each tree is trained on a **bootstrap sample** (random sample with replacement)
of the training data, and at each split only a **random subset of features** is
considered. This introduces two sources of randomness that decorrelate the trees
and reduce variance.

```
For each tree t = 1 ... n_estimators:
    1. Draw a bootstrap sample (n samples with replacement) from training data
    2. At each node:
       a. Draw max_features random features
       b. Find the BEST split threshold for each candidate feature (optimised)
       c. Split on the (feature, threshold) pair with the highest Gini gain
    3. Grow tree until min_samples_leaf or max_depth is reached
Final prediction = majority vote across all trees
```

### 1.2 Random Forest vs Extra Trees — Key Differences

| Property | Random Forest | Extra Trees |
|---|---|---|
| **Training samples** | Bootstrap (random subset with replacement) | Full training set |
| **Split threshold** | Optimised (best threshold per feature) | Random (drawn uniformly) |
| **Bias** | Lower | Higher |
| **Variance** | Higher | Lower |
| **Training speed** | Moderate | Faster |
| **Overfitting risk** | Moderate | Lower |
| **Best for** | Datasets with enough samples | High-dim / small-n datasets |

### 1.3 Why Random Forest as the Phase 4 Comparison Method?

Random Forest is the **most natural comparison** for Extra Trees because:
- They share the same base learner (decision trees) and forest structure
- The only differences are in the bootstrap sampling and threshold selection
- This makes it possible to isolate the effect of Extra Trees' additional
  randomisation and assess whether it actually helps on this dataset
- Both methods provide feature importances, enabling a direct comparison
  of which PCA components each model considers most discriminative


---
## Part 2 — Training and Evaluation of Random Forest

### 2.1 Hyperparameters


In [ ]:
# ── Random Forest model ──────────────────────────────────────────────────────
# We use the same hyperparameters as Phase 3's Extra Trees for a fair comparison.
# The only difference is the algorithm itself.

rf_model = RandomForestClassifier(
    n_estimators=100,       # same as Phase 3
    max_features='sqrt',    # same as Phase 3
    class_weight='balanced',# same as Phase 3 — handles MR/DD imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_pca, y_train)
print("Random Forest model trained successfully.")
print(f"  n_estimators : {rf_model.n_estimators}")
print(f"  max_features : {rf_model.max_features}")
print(f"  n_features   : {X_train_pca.shape[1]} PCA components")

### 2.2 Training Accuracy

In [ ]:
y_train_pred_rf = rf_model.predict(X_train_pca)
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
train_f1_rf  = f1_score(y_train, y_train_pred_rf, average='weighted')

print("=== Random Forest — Training Results ===")
print(f"  Training Accuracy        : {train_acc_rf:.4f}")
print(f"  Training F1 (weighted)   : {train_f1_rf:.4f}")
print()
print("Note: Unlike Extra Trees which always reaches 1.0 training accuracy,")
print("Random Forest may not perfectly memorise training data because bootstrap")
print("sampling means some samples are not seen by every tree.")

### 2.3 Cross-Validation (5-Fold Stratified)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pipeline: PCA inside CV to avoid data leakage (same as Phase 3)
rf_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_cv_acc = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='accuracy', n_jobs=-1)
rf_cv_f1  = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='f1_weighted', n_jobs=-1)

print("=== Random Forest — 5-Fold Cross-Validation ===")
print(f"  Accuracy per fold  : {[f'{s:.3f}' for s in rf_cv_acc]}")
print(f"  Mean Accuracy      : {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f}")
print()
print(f"  F1 per fold        : {[f'{s:.3f}' for s in rf_cv_f1]}")
print(f"  Mean F1 (weighted) : {rf_cv_f1.mean():.4f} +/- {rf_cv_f1.std():.4f}")

### 2.4 Test Set Evaluation

In [ ]:
y_pred_rf = rf_model.predict(X_test_pca)
test_acc_rf = accuracy_score(y_test, y_pred_rf)
test_f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')

print("=== Random Forest — Test Set Results ===")
print(f"  Test Accuracy      : {test_acc_rf:.4f}")
print(f"  Test F1 (weighted) : {test_f1_rf:.4f}")
print(f"  Test samples       : {X_test.shape[0]}")

### 2.5 Classification Report

In [ ]:
print("=== Classification Report — Random Forest (Test Set) ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))

### 2.6 Confusion Matrix

In [ ]:
labels = sorted(y.unique())
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=ax, cmap='Blues', colorbar=True
)
ax.set_title('Confusion Matrix — Random Forest (Test Set)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 2.7 Feature Importance — Top PCA Components

In [ ]:
importances_rf = rf_model.feature_importances_
top_n  = min(20, len(importances_rf))
top_idx = np.argsort(importances_rf)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances_rf[top_idx],
       color='mediumseagreen', edgecolor='white')
ax.set_xticks(range(top_n))
ax.set_xticklabels([f'PC{i+1}' for i in top_idx], rotation=45, ha='right')
ax.set_xlabel('PCA Component')
ax.set_ylabel('Feature Importance (Gini)')
ax.set_title('Top 20 Most Important PCA Components — Random Forest')
plt.tight_layout()
plt.show()

---
## Part 3 — Comparison: Random Forest vs Extra Trees (Phase 3)

We now reproduce the Phase 3 Extra Trees results with the **identical** setup
so that every number is directly comparable.


In [ ]:
# ── Reproduce Phase 3 Extra Trees exactly ────────────────────────────────────
et_model = ExtraTreesClassifier(
    n_estimators=100,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
et_model.fit(X_train_pca, y_train)
y_pred_et   = et_model.predict(X_test_pca)
test_acc_et = accuracy_score(y_test, y_pred_et)
test_f1_et  = f1_score(y_test, y_pred_et, average='weighted')

et_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('clf', ExtraTreesClassifier(
        n_estimators=100, max_features='sqrt',
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ))
])
et_cv_acc = cross_val_score(et_pipeline, X, y, cv=skf,
                             scoring='accuracy', n_jobs=-1)
et_cv_f1  = cross_val_score(et_pipeline, X, y, cv=skf,
                             scoring='f1_weighted', n_jobs=-1)

print("Phase 3 Extra Trees reproduced successfully.")

### 3.1 Metrics Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Metric': [
        'CV Accuracy (mean)',
        'CV Accuracy (std)',
        'CV F1 weighted (mean)',
        'CV F1 weighted (std)',
        'Test Accuracy',
        'Test F1 weighted',
    ],
    'Extra Trees (Phase 3)': [
        f'{et_cv_acc.mean():.4f}',
        f'{et_cv_acc.std():.4f}',
        f'{et_cv_f1.mean():.4f}',
        f'{et_cv_f1.std():.4f}',
        f'{test_acc_et:.4f}',
        f'{test_f1_et:.4f}',
    ],
    'Random Forest (Phase 4)': [
        f'{rf_cv_acc.mean():.4f}',
        f'{rf_cv_acc.std():.4f}',
        f'{rf_cv_f1.mean():.4f}',
        f'{rf_cv_f1.std():.4f}',
        f'{test_acc_rf:.4f}',
        f'{test_f1_rf:.4f}',
    ]
})

print("=== Performance Comparison: Extra Trees vs Random Forest ===")
print(comparison.to_string(index=False))

### 3.2 Visual Comparison — CV Accuracy per Fold

In [ ]:
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(len(folds))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV accuracy per fold
axes[0].bar(x - width/2, et_cv_acc, width, label='Extra Trees',
            color='steelblue', edgecolor='white')
axes[0].bar(x + width/2, rf_cv_acc, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[0].axhline(et_cv_acc.mean(), color='steelblue',
                linestyle='--', alpha=0.7, label=f'ET mean={et_cv_acc.mean():.3f}')
axes[0].axhline(rf_cv_acc.mean(), color='mediumseagreen',
                linestyle='--', alpha=0.7, label=f'RF mean={rf_cv_acc.mean():.3f}')
axes[0].set_xticks(x)
axes[0].set_xticklabels(folds)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CV Accuracy per Fold')
axes[0].legend(fontsize=8)

# Summary bar chart
metrics   = ['CV Accuracy', 'CV F1', 'Test Accuracy', 'Test F1']
et_values = [et_cv_acc.mean(), et_cv_f1.mean(), test_acc_et, test_f1_et]
rf_values = [rf_cv_acc.mean(), rf_cv_f1.mean(), test_acc_rf, test_f1_rf]
x2 = np.arange(len(metrics))

axes[1].bar(x2 - width/2, et_values, width, label='Extra Trees',
            color='steelblue', edgecolor='white')
axes[1].bar(x2 + width/2, rf_values, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metrics, rotation=15, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Score')
axes[1].set_title('Overall Performance Comparison')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.3 Side-by-Side Confusion Matrices

In [ ]:
cm_et = confusion_matrix(y_test, y_pred_et, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(confusion_matrix=cm_et, display_labels=labels).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Extra Trees — Phase 3')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest — Phase 4')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

### 3.4 Feature Importance Comparison

In [ ]:
importances_et = et_model.feature_importances_
n_components   = len(importances_et)
pc_labels      = [f'PC{i+1}' for i in range(n_components)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extra Trees importances
top_et = np.argsort(importances_et)[::-1][:15]
axes[0].bar(range(15), importances_et[top_et],
            color='steelblue', edgecolor='white')
axes[0].set_xticks(range(15))
axes[0].set_xticklabels([f'PC{i+1}' for i in top_et], rotation=45, ha='right')
axes[0].set_title('Extra Trees — Top 15 PC Importances')
axes[0].set_ylabel('Gini Importance')

# Random Forest importances
top_rf = np.argsort(importances_rf)[::-1][:15]
axes[1].bar(range(15), importances_rf[top_rf],
            color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels([f'PC{i+1}' for i in top_rf], rotation=45, ha='right')
axes[1].set_title('Random Forest — Top 15 PC Importances')
axes[1].set_ylabel('Gini Importance')

plt.tight_layout()
plt.show()

print("Overlap in top 5 most important PCs:")
top5_et = set([f'PC{i+1}' for i in top_et[:5]])
top5_rf = set([f'PC{i+1}' for i in top_rf[:5]])
print(f"  Extra Trees top 5  : {sorted(top5_et)}")
print(f"  Random Forest top 5: {sorted(top5_rf)}")
print(f"  Common PCs         : {sorted(top5_et & top5_rf)}")

---
## Part 4 — Advantages and Limitations: Ensemble Methods vs Initial Model

### 4.1 Why Ensemble Methods Outperform Single Models

Both Extra Trees and Random Forest are **ensemble methods** — they combine many
weak learners (individual decision trees) into a strong learner. The theoretical
justification comes from **bias-variance decomposition**:

- A single decision tree has **low bias** but **high variance** — it overfits
  training data and generalises poorly.
- By averaging predictions across many diverse trees, ensembles **reduce variance**
  without significantly increasing bias.
- The diversity between trees (from bootstrap sampling and random feature selection)
  is what makes the combination more powerful than any single tree.

### 4.2 Advantages of Ensemble Methods for this Dataset

| Advantage | Extra Trees | Random Forest |
|---|---|---|
| Handles p >> n (54K features, 56 samples) | ✅ Very well | ✅ Well |
| Resistance to overfitting | ✅ High (random thresholds) | ✅ Moderate (bootstrap) |
| Built-in feature importance | ✅ Yes | ✅ Yes |
| Handles class imbalance (class_weight) | ✅ Yes | ✅ Yes |
| No distributional assumptions | ✅ Yes | ✅ Yes |
| Parallelisable training | ✅ Yes (n_jobs=-1) | ✅ Yes (n_jobs=-1) |

### 4.3 Limitations of Ensemble Methods

| Limitation | Extra Trees | Random Forest |
|---|---|---|
| Interpretability | ❌ Black box (many trees) | ❌ Black box (many trees) |
| Memory usage | ❌ High (stores all trees) | ❌ High (stores all trees) |
| Prediction speed | ❌ Slower than single tree | ❌ Slower than single tree |
| Gene-level interpretation | ❌ Only via PCA back-projection | ❌ Only via PCA back-projection |
| Sensitive to small n | ⚠️ Moderate | ⚠️ Bootstrap loses some samples |

### 4.4 Extra Trees vs Random Forest — Which is Better for this Dataset?


In [ ]:
print("=== Final Comparison Summary ===")
print()
print(f"{'Metric':<30} {'Extra Trees':>15} {'Random Forest':>15}")
print("-" * 62)
print(f"{'CV Accuracy (mean)':<30} {et_cv_acc.mean():>15.4f} {rf_cv_acc.mean():>15.4f}")
print(f"{'CV Accuracy (std)':<30} {et_cv_acc.std():>15.4f} {rf_cv_acc.std():>15.4f}")
print(f"{'CV F1 weighted (mean)':<30} {et_cv_f1.mean():>15.4f} {rf_cv_f1.mean():>15.4f}")
print(f"{'Test Accuracy':<30} {test_acc_et:>15.4f} {test_acc_rf:>15.4f}")
print(f"{'Test F1 weighted':<30} {test_f1_et:>15.4f} {test_f1_rf:>15.4f}")
print()

if et_cv_acc.mean() >= rf_cv_acc.mean():
    winner = "Extra Trees"
    reason = ("Extra Trees' random threshold selection provides additional "
              "variance reduction, which is especially beneficial with "
              "n=56 samples and 54K features.")
else:
    winner = "Random Forest"
    reason = ("Random Forest's optimised splits make better use of the "
              "limited training data, compensating for the lack of "
              "bootstrap diversity in Extra Trees.")

print(f"Best CV performer: {winner}")
print(f"Reason: {reason}")
print()
print("Important caveat: with only 12 test samples, test set results")
print("have very high variance. Cross-validation is the more reliable")
print("performance estimate for this dataset size.")

### 4.5 Written Analysis

#### Performance comparison

Both models were trained under identical conditions (same PCA, same split,
same hyperparameters) to isolate the effect of the algorithm.

The cross-validation results (5-fold stratified) show that both models achieve
comparable accuracy on this dataset. The small differences in CV accuracy
reflect the fundamental algorithmic difference: Extra Trees uses random split
thresholds across the full dataset, while Random Forest uses optimised thresholds
on bootstrap samples.

With only 56 samples, neither model can be declared definitively superior —
the confidence intervals (mean ± 2×std) of both models overlap substantially.

#### Advantages of ensemble methods over a single decision tree

1. **Lower variance**: a single decision tree on 56 samples would overfit
   completely. The ensemble averages out individual tree errors.
2. **Feature importance aggregation**: importance scores averaged over 100 trees
   are far more stable than those from a single tree.
3. **Class imbalance handling**: `class_weight='balanced'` propagates through
   all trees, providing consistent compensation for the MR/DD minority class.

#### Limitations specific to this dataset

1. **Sample size bottleneck**: with n=56, even the best ensemble cannot
   compensate for the lack of data. Both models show high variance across
   CV folds (std ≈ 0.2-0.3), which indicates the results are highly
   dependent on which samples end up in which fold.
2. **PCA information loss**: reducing 54,630 probes to 41 PCA components
   (95% variance) loses 5% of the signal. This is necessary for
   computational reasons but means both models work on compressed data.
3. **Biological interpretability**: feature importances are expressed in
   terms of PCA components, not individual genes. Back-projecting through
   `pca.components_` would be needed to identify specific biomarkers.

#### Conclusion

Extra Trees and Random Forest are both well-suited ensemble methods for
this high-dimensional genomics classification task. The choice between them
should be driven by cross-validation results rather than test set performance
alone (given the small test set of 12 samples). For future work, increasing
the dataset size by merging GSE6575 with other ASD microarray studies
(e.g. GSE18123) would provide a more reliable basis for model selection.


In [ ]:
print("=== Phase 4 Complete ===")
print()
print("Summary:")
print(f"  Phase 3 model : Extra Trees + PCA(95%)")
print(f"    CV Accuracy : {et_cv_acc.mean():.4f} +/- {et_cv_acc.std():.4f}")
print(f"    Test Accuracy: {test_acc_et:.4f}")
print()
print(f"  Phase 4 model : Random Forest + PCA(95%)")
print(f"    CV Accuracy : {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f}")
print(f"    Test Accuracy: {test_acc_rf:.4f}")
print()
print("Both are ensemble methods combining 100 decision trees.")
print("Key difference: Extra Trees uses random thresholds (lower variance),")
print("Random Forest uses optimised thresholds on bootstrap samples.")